# Structural Exclusion and Hypertransmission Segmentation

This notebook investigates an alternative approach to hypertransmission and
barcoding detection.

Rather than immediately combining several standardized features into a weighted
score, the workflow separates structural information from intensity information.

The experimental sequence is:

1. Load and preprocess a selected OCT B-scan.
2. Calculate simple gradient-based verticality.
3. Construct a structural-pixel exclusion mask.
4. Estimate the intensity distribution of the remaining choroidal pixels.
5. Calculate column-level mean, median, and upper-quantile intensity.
6. Construct and visualize candidate hypertransmission regions.
7. Refine candidate regions using depth continuity and vertical organization.

The purpose of this notebook is diagnostic. Thresholds and segmentation rules
are exploratory and are not yet clinically validated.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError(
            "Could not locate the project root."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

from src.detector.data import build_grouped_volume_registry, load_e2e_volume
from src.detector.preprocessing import preprocess_bscan
from src.detector.features import compute_column_intensity_statistics, compute_simple_verticality_map, create_structural_mask
print("Project root:", PROJECT_ROOT)

## Volume and B-Scan Selection

A single volume and explicit zero-based B-scan index are selected for the first
diagnostic experiment.

The same notebook can later be rerun on other subjects and scans by changing
only `SUBJECT_ID` and `BSCAN_INDEX`.

In [ ]:
E2E_DIRECTORY = PROJECT_ROOT / "data"  / "heyex" / "meta"

PROGRESSION_GROUPS = {
    "fast": [8,9,12,41,49],
    "slow": [17,23,35,36,47],
}

registry = build_grouped_volume_registry(
    e2e_directory=E2E_DIRECTORY,
    progression_groups=PROGRESSION_GROUPS,
)

print( "Registry entries:",len(registry))

In [ ]:
SUBJECT_ID, BSCAN_INDEX = 8, 48

In [ ]:
matching_records = [
    record
    for record in registry
    if int(record.subject_id) == int(SUBJECT_ID)
]

if not matching_records:
    raise KeyError( f"Subject {SUBJECT_ID} was not found in the volume registry.")

volume_record = matching_records[0]
volume = load_e2e_volume(volume_record.e2e_path)

if not ( 0 <= BSCAN_INDEX< len(volume)):
    raise IndexError(
        f"BSCAN_INDEX={BSCAN_INDEX} is outside "
        f"the valid range 0 to {len(volume) - 1}."
    )

print("Subject:", volume_record.subject_id)
print("Progression group:",volume_record.progression_group,)
print("Number of B-scans:",len(volume),)
print("Selected B-scan:",BSCAN_INDEX,)
print("Volume shape:",volume.shape,)

## Preprocessing

The selected B-scan is:

1. flattened to Bruch's membrane;
2. cropped to a 150-pixel region below BM;
3. normalized using whole-ROI z-score normalization;
4. denoised using an anisotropic Gaussian filter.

The Gaussian standard deviations are 1.0 pixels through depth and 0.5 pixels
horizontally. This reduces speckle while limiting horizontal blurring of narrow
vertical structures.

In [ ]:
PREPROCESSING_CONFIG = {
    "layer_name": "BM",

    # Flattening
    "reference_row": None,
    "flatten_fill_value": 0.0,

    # Sub-layer crop
    "depth_below_layer": 150,
    "include_boundary": True,
    "require_full_depth": False,
    "crop_fill_value": 0.0,

    # Whole-ROI normalization
    "normalization_method": "zscore",
    "lower_percentile": 1.0,
    "upper_percentile": 99.0,

    # Anisotropic Gaussian denoising
    "denoise_method": "gaussian",
    "gaussian_sigma": (
        1.0, # z
        0.5, # x
    ),
}

In [ ]:
processed = preprocess_bscan(
    volume=volume,
    bscan_index=BSCAN_INDEX,
    **PREPROCESSING_CONFIG,
)

print("Raw shape:",processed.raw_bscan.shape)
print("Flattened shape:",processed.flattened_bscan.shape)
print("Sub-layer crop shape:",processed.sub_layer_crop.shape)
print("Normalized shape:",processed.normalized_scan.shape)
print("Denoised shape:",processed.denoised_scan.shape)

In [ ]:
fig, axes = plt.subplots(
    nrows=5,
    ncols=1,
    figsize=(14, 18),
)

stages = [
    (processed.raw_bscan, "Raw B-scan"),
    (processed.flattened_bscan, "Flattened to BM"),
    (processed.sub_layer_crop, "150-pixel sub-BM crop"),
    (processed.normalized_scan, "Whole-ROI z-score normalization"),
    (processed.denoised_scan, "Gaussian-denoised scan")
    ]

for axis, (image,title) in zip( axes,stages):
    axis.imshow(
        image,
        cmap="gray",
        aspect="auto",
    )
    axis.set_title(title)
    axis.set_xlabel("Horizontal position")
    axis.set_ylabel("Axial position")
    
plt.tight_layout()
plt.show()

## Simple Gradient-Based Verticality

A vertically oriented structure produces a strong change in intensity when
moving horizontally across it, while changing less when moving along its depth.

The simple verticality score is

$$
V(z,x)
=
\frac{
|I_x(z,x)|
}{
|I_x(z,x)|+|I_z(z,x)|+\varepsilon
},
$$

where:

- $I_x$ is the horizontal image gradient;
- $I_z$ is the depth-wise image gradient;
- $\varepsilon$ prevents division by zero.

Values near one indicate horizontal-gradient dominance and therefore stronger
evidence of vertical structure. Values near zero indicate depth-wise-gradient
dominance.

In [ ]:
STRUCTURAL_CONFIG = {
    "simple_verticality": {
        "smoothing_sigma": 1.0,
    },

    "structural_mask": {
        "verticality_threshold": 0.60,

        # Disabled for the first raw inspection. (None)
        "minimum_gradient_magnitude": None,

        # Disabled until the uncleaned mask is evaluated. (0)
        "minimum_component_size": 0,
    },

    "column_statistics": {
        "upper_quantile": 0.90,
        "minimum_valid_pixels": 5,
    },
}

In [ ]:
simple_verticality_map, simple_verticality_diagnostics = compute_simple_verticality_map(
    image=processed.denoised_scan,
    **STRUCTURAL_CONFIG["simple_verticality"],
)

print("Verticality map shape:",simple_verticality_map.shape)
print("Verticality range:",
    float(simple_verticality_map.min()),
    "to",
    float(simple_verticality_map.max()),
)
print("Mean verticality:",float( simple_verticality_map.mean()),)

In [ ]:
gradient_magnitude = np.hypot(
    simple_verticality_diagnostics["gradient_x"],
    simple_verticality_diagnostics["gradient_z"],
).astype(np.float32)

print("Gradient magnitude range:",
    float(gradient_magnitude.min()),
    "to",
    float(gradient_magnitude.max()),
)

In [ ]:
fig, axes = plt.subplots(
    nrows=4,
    ncols=1,
    figsize=(14, 14),
    sharex=True,
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
)

axes[0].set_title("Denoised sub-BM scan")
axes[0].set_ylabel("Depth below BM")

gradient_x_image = axes[1].imshow(
    np.abs(simple_verticality_diagnostics["gradient_x"]),
    cmap="magma",
    aspect="auto",
)

axes[1].set_title("Absolute horizontal gradient")
axes[1].set_ylabel("Depth below BM")

fig.colorbar(
    gradient_x_image,
    ax=axes[1],
    label="Gradient magnitude",
)

gradient_z_image = axes[2].imshow(
    np.abs(simple_verticality_diagnostics["gradient_z"]),
    cmap="magma",
    aspect="auto",
)

axes[2].set_title("Absolute depth-wise gradient")

axes[2].set_ylabel("Depth below BM")

fig.colorbar(
    gradient_z_image,
    ax=axes[2],
    label="Gradient magnitude",
)

verticality_image = axes[3].imshow(
    simple_verticality_map,
    cmap="viridis",
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
)

axes[3].set_title("Simple gradient-based verticality")
axes[3].set_xlabel("Horizontal position")
axes[3].set_ylabel("Depth below BM")

fig.colorbar(
    verticality_image,
    ax=axes[3],
    label="Verticality",
)

plt.tight_layout()
plt.show()

## Structural-Pixel Exclusion Mask

Pixels exceeding a selected verticality threshold are marked as structurally
organized:

$$
M_{\mathrm{structure}}(z,x)
=
\mathbf{1}
\left[
V(z,x)\geq \tau_V
\right].
$$

These pixels are not immediately classified as barcoding. Instead, they are
temporarily excluded so that the remaining pixel intensities can provide an
estimate of the non-structural choroidal background.

The initial threshold is exploratory:

$$
\tau_V=0.60.
$$

In [ ]:
(
    structural_mask,
    structural_mask_metadata,
) = create_structural_mask(
    verticality_map=(
        simple_verticality_map
    ),
    **STRUCTURAL_CONFIG[
        "structural_mask"
    ],
)

print("Structural-pixel fraction:",structural_mask_metadata["structural_pixel_fraction"],)
print("Structural-pixel count:",structural_mask_metadata["structural_pixel_count"],)

In [ ]:
structurally_excluded_scan = (
    processed.denoised_scan.copy()
)

structurally_excluded_scan[
    structural_mask
] = np.nan

fig, axes = plt.subplots(
    nrows=4,
    ncols=1,
    figsize=(14, 14),
    sharex=True,
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
)

axes[0].set_title(
    "Denoised sub-BM scan"
)

axes[0].set_ylabel(
    "Depth below BM"
)

axes[1].imshow(
    simple_verticality_map,
    cmap="viridis",
    aspect="auto",
    vmin=0.0,
    vmax=1.0,
)

axes[1].set_title(
    "Simple verticality map"
)

axes[1].set_ylabel(
    "Depth below BM"
)

axes[2].imshow(
    structural_mask,
    cmap="binary",
    aspect="auto",
    interpolation="nearest",
)

axes[2].set_title(
    "Structural-pixel exclusion mask"
)

axes[2].set_ylabel(
    "Depth below BM"
)

masked_colormap = plt.get_cmap(
    "gray"
).copy()

masked_colormap.set_bad(
    color="tab:red",
    alpha=0.65,
)

axes[3].imshow(
    structurally_excluded_scan,
    cmap=masked_colormap,
    aspect="auto",
)

axes[3].set_title(
    "Denoised scan with structural pixels highlighted"
)

axes[3].set_xlabel(
    "Horizontal position"
)

axes[3].set_ylabel(
    "Depth below BM"
)

plt.tight_layout()
plt.show()

## Intensity Distribution After Structural Exclusion

The non-structural choroid is defined as the set of pixels not included in the
structural mask:

$$
\mathcal{C}
=
\left\{
I(z,x):
M_{\mathrm{structure}}(z,x)=0
\right\}.
$$

The full ROI distribution is compared with the structurally cleaned
distribution.

The following summaries are inspected:

- mean;
- median;
- 90th percentile;
- interquartile range.

This determines whether removing strongly oriented pixels changes the estimated
background-intensity distribution.

In [ ]:
all_roi_intensities = (
    processed.denoised_scan[
        np.isfinite(
            processed.denoised_scan
        )
    ]
)

clean_choroid_intensities = (
    processed.denoised_scan[
        ~structural_mask
    ]
)

print(
    "All ROI pixels:",
    all_roi_intensities.size,
)

print(
    "Retained clean-choroid pixels:",
    clean_choroid_intensities.size,
)

print(
    "Retained fraction:",
    clean_choroid_intensities.size
    / all_roi_intensities.size,
)

In [ ]:
def summarize_intensity_distribution(
    values: np.ndarray,
) -> dict[str, float]:
    """
    Return robust intensity summaries.
    """
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    values = values[
        np.isfinite(values)
    ]

    return {
        "mean": float(
            np.mean(values)
        ),
        "median": float(
            np.median(values)
        ),
        "q25": float(
            np.quantile(
                values,
                0.25,
            )
        ),
        "q75": float(
            np.quantile(
                values,
                0.75,
            )
        ),
        "q90": float(
            np.quantile(
                values,
                0.90,
            )
        ),
        "q95": float(
            np.quantile(
                values,
                0.95,
            )
        ),
        "standard_deviation": float(
            np.std(values)
        ),
    }


all_roi_summary = (
    summarize_intensity_distribution(
        all_roi_intensities
    )
)

clean_choroid_summary = (
    summarize_intensity_distribution(
        clean_choroid_intensities
    )
)

print("All ROI:")
for name, value in (
    all_roi_summary.items()
):
    print(
        f"  {name}: {value:.4f}"
    )

print("\nStructurally cleaned choroid:")
for name, value in (
    clean_choroid_summary.items()
):
    print(
        f"  {name}: {value:.4f}"
    )

In [ ]:
combined_minimum = float(
    min(
        all_roi_intensities.min(),
        clean_choroid_intensities.min(),
    )
)

combined_maximum = float(
    max(
        all_roi_intensities.max(),
        clean_choroid_intensities.max(),
    )
)

histogram_bins = np.linspace(
    combined_minimum,
    combined_maximum,
    100,
)

plt.figure(
    figsize=(13, 6)
)

plt.hist(
    all_roi_intensities,
    bins=histogram_bins,
    density=True,
    alpha=0.45,
    label="All ROI pixels",
)

plt.hist(
    clean_choroid_intensities,
    bins=histogram_bins,
    density=True,
    alpha=0.45,
    label=(
        "After structural exclusion"
    ),
)

plt.axvline(
    clean_choroid_summary[
        "median"
    ],
    linestyle="--",
    linewidth=1.5,
    label="Clean median",
)

plt.axvline(
    clean_choroid_summary[
        "q90"
    ],
    linestyle=":",
    linewidth=1.8,
    label="Clean 90th percentile",
)

plt.title(
    "Intensity distribution before and after structural exclusion"
)

plt.xlabel(
    "Normalized denoised intensity"
)

plt.ylabel(
    "Density"
)

plt.legend()
plt.grid(
    alpha=0.20
)

plt.tight_layout()
plt.show()

## Column-Level Intensity Summaries

For each horizontal position $x$, only non-structural pixels are retained.

The following statistics are calculated through depth:

$$
\mu(x)
=
\operatorname{mean}
\left(
\mathcal{C}_x
\right),
$$

$$
m(x)
=
\operatorname{median}
\left(
\mathcal{C}_x
\right),
$$

$$
q_{90}(x)
=
Q_{0.90}
\left(
\mathcal{C}_x
\right),
$$

where $\mathcal{C}_x$ is the set of retained intensities in column $x$.

The mean reflects broad intensity elevation, the median describes the typical
column intensity, and the 90th percentile reflects the upper-intensity portion
of the column.

In [ ]:
(
    column_statistics,
    column_statistics_metadata,
) = compute_column_intensity_statistics(
    image=processed.denoised_scan,
    exclusion_mask=structural_mask,
    **STRUCTURAL_CONFIG[
        "column_statistics"
    ],
)

print(
    "Columns with sufficient data:",
    column_statistics_metadata[
        "columns_with_sufficient_data"
    ],
)

print(
    "Columns with insufficient data:",
    column_statistics_metadata[
        "columns_with_insufficient_data"
    ],
)

In [ ]:
horizontal_positions = np.arange(
    processed.denoised_scan.shape[1]
)

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(14, 10),
    sharex=True,
)

axes[0].plot(
    horizontal_positions,
    column_statistics[
        "mean"
    ],
    linewidth=1.1,
)

axes[0].set_title(
    "Mean intensity after structural exclusion"
)

axes[0].set_ylabel(
    "Mean"
)

axes[1].plot(
    horizontal_positions,
    column_statistics[
        "median"
    ],
    linewidth=1.1,
)

axes[1].set_title(
    "Median intensity after structural exclusion"
)

axes[1].set_ylabel(
    "Median"
)

axes[2].plot(
    horizontal_positions,
    column_statistics[
        "upper_quantile"
    ],
    linewidth=1.1,
)

axes[2].set_title(
    "90th-percentile intensity after structural exclusion"
)

axes[2].set_xlabel(
    "Horizontal position"
)

axes[2].set_ylabel(
    "90th percentile"
)

for axis in axes:
    axis.grid(
        alpha=0.25
    )

    axis.set_xlim(
        0,
        processed.denoised_scan.shape[1]
        - 1,
    )

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(14, 6)
)

plt.plot(
    horizontal_positions,
    column_statistics[
        "mean"
    ],
    label="Mean",
    linewidth=1.2,
)

plt.plot(
    horizontal_positions,
    column_statistics[
        "median"
    ],
    label="Median",
    linewidth=1.2,
)

plt.plot(
    horizontal_positions,
    column_statistics[
        "upper_quantile"
    ],
    label="90th percentile",
    linewidth=1.2,
)

plt.title(
    "Column intensity summaries after structural exclusion"
)

plt.xlabel(
    "Horizontal position"
)

plt.ylabel(
    "Normalized intensity"
)

plt.xlim(
    0,
    processed.denoised_scan.shape[1]
    - 1,
)

plt.legend()
plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

Our simple verticality ratio is labeling too many pixels since we are looking at relative direction of gradients. The two distributions nearly overlap so the current mask is not removing s distinct structural population. We will add a gradient-magnitude gate, which will require a pixel to have both a vertically compatible gradient direction and a sufficiently strong boundary. 
$$
M_{\mathrm{structure}}(z,x)
=
\mathbf{1}
\left[
V(z,x)\geq\tau_V
\right]
\mathbf{1}
\left[
G(z,x)\geq\tau_G
\right],
$$
where
$$
G(z,x)
=
\sqrt{
I_x(z,x)^2+I_z(z,x)^2
}.
$$

In [ ]:
# look at grad-mag dist'n
gradient_quantiles = {
    quantile: float(
        np.quantile(
            gradient_magnitude,
            quantile,
        )
    )
    for quantile in (
        0.50,
        0.60,
        0.70,
        0.75,
        0.80,
        0.85,
        0.90,
        0.95,
    )
}

for quantile, value in gradient_quantiles.items():
    print(
        f"Gradient Q{int(100 * quantile):02d}: "
        f"{value:.6f}"
    )

In [ ]:
STRUCTURAL_MASK_TESTS = {
    "verticality_only": {
        "verticality_threshold": 0.60,
        "minimum_gradient_magnitude": None,
    },
    "gradient_q70": {
        "verticality_threshold": 0.60,
        "minimum_gradient_magnitude": (
            gradient_quantiles[0.70]
        ),
    },
    "gradient_q80": {
        "verticality_threshold": 0.60,
        "minimum_gradient_magnitude": (
            gradient_quantiles[0.80]
        ),
    },
    "gradient_q90": {
        "verticality_threshold": 0.60,
        "minimum_gradient_magnitude": (
            gradient_quantiles[0.90]
        ),
    },
}

In [ ]:
structural_mask_tests = {}

for label, mask_config in (
    STRUCTURAL_MASK_TESTS.items()
):
    test_mask, test_metadata = (
        create_structural_mask(
            verticality_map=(
                simple_verticality_map
            ),
            gradient_magnitude=(
                gradient_magnitude
            ),
            verticality_threshold=(
                mask_config[
                    "verticality_threshold"
                ]
            ),
            minimum_gradient_magnitude=(
                mask_config[
                    "minimum_gradient_magnitude"
                ]
            ),
            minimum_component_size=0,
        )
    )

    structural_mask_tests[
        label
    ] = {
        "mask": test_mask,
        "metadata": test_metadata,
    }

    print(
        f"{label}: "
        f"{test_metadata['structural_pixel_fraction']:.3f} "
        "of pixels excluded"
    )

In [ ]:
fig, axes = plt.subplots(
    nrows=len(
        structural_mask_tests
    ) + 1,
    ncols=1,
    figsize=(14, 14),
    sharex=True,
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
)

axes[0].set_title(
    "Denoised sub-BM scan"
)

axes[0].set_ylabel(
    "Depth below BM"
)

for axis, (
    label,
    test_result,
) in zip(
    axes[1:],
    structural_mask_tests.items(),
):
    axis.imshow(
        test_result["mask"],
        cmap="binary",
        aspect="auto",
        interpolation="nearest",
    )

    excluded_fraction = (
        test_result[
            "metadata"
        ][
            "structural_pixel_fraction"
        ]
    )

    axis.set_title(
        f"{label}: "
        f"{excluded_fraction:.1%} excluded"
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[-1].set_xlabel(
    "Horizontal position"
)

plt.tight_layout()
plt.show()

In [ ]:
SELECTED_STRUCTURAL_MASK = "gradient_q80"

structural_mask = structural_mask_tests[
    SELECTED_STRUCTURAL_MASK
]["mask"]

structural_mask_metadata = structural_mask_tests[
    SELECTED_STRUCTURAL_MASK
]["metadata"]

print(
    "Selected structural mask:",
    SELECTED_STRUCTURAL_MASK,
)

print(
    "Excluded pixel fraction:",
    structural_mask_metadata[
        "structural_pixel_fraction"
    ],
)

In [ ]:
all_roi_intensities = processed.denoised_scan[
    np.isfinite(
        processed.denoised_scan
    )
]

clean_choroid_intensities = processed.denoised_scan[
    ~structural_mask
]

all_roi_summary = summarize_intensity_distribution(
    all_roi_intensities
)

clean_choroid_summary = summarize_intensity_distribution(
    clean_choroid_intensities
)

print("All ROI:")
for name, value in all_roi_summary.items():
    print(
        f"  {name}: {value:.4f}"
    )

print("\nGradient-gated clean choroid:")
for name, value in clean_choroid_summary.items():
    print(
        f"  {name}: {value:.4f}"
    )

In [ ]:
(
    column_statistics,
    column_statistics_metadata,
) = compute_column_intensity_statistics(
    image=processed.denoised_scan,
    exclusion_mask=structural_mask,
    **STRUCTURAL_CONFIG[
        "column_statistics"
    ],
)

print(
    "Columns with sufficient data:",
    column_statistics_metadata[
        "columns_with_sufficient_data"
    ],
)

print(
    "Columns with insufficient data:",
    column_statistics_metadata[
        "columns_with_insufficient_data"
    ],
)

print(
    "Median retained fraction:",
    float(
        np.nanmedian(
            column_statistics[
                "valid_pixel_fraction"
            ]
        )
    ),
)

In [ ]:
# mask out unstable columns
MINIMUM_RETAINED_FRACTION = 0.75

stable_column_mask = (
    column_statistics[
        "valid_pixel_fraction"
    ]
    >= MINIMUM_RETAINED_FRACTION
)


In [ ]:
stable_column_statistics = {}

for statistic_name in (
    "mean",
    "median",
    "upper_quantile",
    "standard_deviation",
):
    values = column_statistics[
        statistic_name
    ].copy()

    values[
        ~stable_column_mask
    ] = np.nan

    stable_column_statistics[
        statistic_name
    ] = values

In [ ]:
horizontal_positions = np.arange(
    processed.denoised_scan.shape[1]
)

fig, axes = plt.subplots(
    5,
    1,
    figsize=(14, 15),
    sharex=True,
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
)

axes[0].set_title(
    "Denoised sub-BM scan"
)

axes[0].set_ylabel(
    "Depth below BM"
)

axes[1].imshow(
    structural_mask,
    cmap="binary",
    aspect="auto",
    interpolation="nearest",
)

axes[1].set_title(
    f"Selected structural mask: "
    f"{SELECTED_STRUCTURAL_MASK}"
)

axes[1].set_ylabel(
    "Depth below BM"
)

axes[2].plot(
    horizontal_positions,
    stable_column_statistics[
        "mean"
    ],
)

axes[2].set_title(
    "Mean intensity after gradient-gated structural exclusion"
)

axes[2].set_ylabel(
    "Mean"
)

axes[3].plot(
    horizontal_positions,
    stable_column_statistics[
        "median"
    ],
)

axes[3].set_title(
    "Median intensity after gradient-gated structural exclusion"
)

axes[3].set_ylabel(
    "Median"
)

axes[4].plot(
    horizontal_positions,
    stable_column_statistics[
        "upper_quantile"
    ],
    label="90th percentile",
)

axes[4].plot(
    horizontal_positions,
    column_statistics[
        "valid_pixel_fraction"
    ],
    label="Retained fraction",
    alpha=0.7,
)

axes[4].axhline(
    MINIMUM_RETAINED_FRACTION,
    linestyle="--",
    label="Minimum retained fraction",
)

axes[4].set_title(
    "Upper intensity and retained-pixel reliability"
)

axes[4].set_xlabel(
    "Horizontal position"
)

axes[4].legend()

for axis in axes[2:]:
    axis.grid(
        alpha=0.25
    )

plt.tight_layout()
plt.show()

## Intensity-Only Hypertransmission Segmentation

After excluding strongly oriented structural pixels, column-level intensity
summaries are used to identify candidate hypertransmission regions.

The median is treated as the primary signal because it increases only when a
substantial portion of a column is brighter than the cleaned choroidal
background. The 90th percentile is used as supporting evidence that the upper
portion of the column is also elevated.

Before thresholding, the column signals are smoothed horizontally to reduce
single-column fluctuations while preserving broader contiguous regions.

Several exploratory rules are compared:

1. Median above a cleaned-background threshold.
2. Median above its scan-level upper quantile.
3. Joint elevation of the median and 90th percentile.
4. Joint elevation with a retained-pixel reliability requirement.

These masks represent candidate hypertransmission only. Verticality and depth
continuity will be applied later to refine hypertransmission into candidate
barcoding regions.

In [ ]:
from scipy.ndimage import gaussian_filter1d

INTENSITY_SEGMENTATION_CONFIG = {
    # Smooth column signals horizontally before thresholding.
    "smoothing_sigma": 2.0,

    # Columns must retain at least this fraction of pixels.
    "minimum_retained_fraction": 0.75,

    # Scan-level quantiles used for threshold comparisons.
    "median_quantile": 0.75,
    "upper_quantile_quantile": 0.75,

    # Spatial cleanup.
    "minimum_positive_run": 5,
    "maximum_negative_gap": 2,

    # Ignore likely edge artifacts.
    "edge_margin": 10,
}

In [ ]:
column_mean = np.asarray(
    column_statistics["mean"],
    dtype=np.float32,
)

column_median = np.asarray(
    column_statistics["median"],
    dtype=np.float32,
)

column_q90 = np.asarray(
    column_statistics["upper_quantile"],
    dtype=np.float32,
)

retained_fraction = np.asarray(
    column_statistics["valid_pixel_fraction"],
    dtype=np.float32,
)

reliable_columns = (
    retained_fraction
    >= INTENSITY_SEGMENTATION_CONFIG[
        "minimum_retained_fraction"
    ]
)


def smooth_finite_signal(
    values: np.ndarray,
    sigma: float,
) -> np.ndarray:
    """
    Smooth a one-dimensional signal while handling any non-finite values.
    """
    values = np.asarray(
        values,
        dtype=np.float32,
    )

    finite = np.isfinite(values)

    if not finite.any():
        raise ValueError(
            "The signal contains no finite values."
        )

    filled = values.copy()

    if not finite.all():
        x = np.arange(
            values.size
        )

        filled[
            ~finite
        ] = np.interp(
            x[~finite],
            x[finite],
            values[finite],
        )

    return gaussian_filter1d(
        filled,
        sigma=float(sigma),
        mode="nearest",
    ).astype(np.float32)


column_mean_smoothed = smooth_finite_signal(
    column_mean,
    INTENSITY_SEGMENTATION_CONFIG[
        "smoothing_sigma"
    ],
)

column_median_smoothed = smooth_finite_signal(
    column_median,
    INTENSITY_SEGMENTATION_CONFIG[
        "smoothing_sigma"
    ],
)

column_q90_smoothed = smooth_finite_signal(
    column_q90,
    INTENSITY_SEGMENTATION_CONFIG[
        "smoothing_sigma"
    ],
)

In [ ]:
image_width = processed.denoised_scan.shape[1]

edge_margin = INTENSITY_SEGMENTATION_CONFIG[
    "edge_margin"
]

threshold_reference_mask = reliable_columns.copy()

threshold_reference_mask[
    :edge_margin
] = False

threshold_reference_mask[
    -edge_margin:
] = False

median_reference_values = (
    column_median_smoothed[
        threshold_reference_mask
    ]
)

q90_reference_values = (
    column_q90_smoothed[
        threshold_reference_mask
    ]
)

median_iqr = float(
    np.quantile(
        median_reference_values,
        0.75,
    )
    - np.quantile(
        median_reference_values,
        0.25,
    )
)

clean_background_median_threshold = (
    clean_choroid_summary["median"]
    + median_iqr
)

scan_median_quantile_threshold = float(
    np.quantile(
        median_reference_values,
        INTENSITY_SEGMENTATION_CONFIG[
            "median_quantile"
        ],
    )
)

scan_q90_quantile_threshold = float(
    np.quantile(
        q90_reference_values,
        INTENSITY_SEGMENTATION_CONFIG[
            "upper_quantile_quantile"
        ],
    )
)

print(
    "Clean-background median + column IQR:",
    clean_background_median_threshold,
)

print(
    "Column-median quantile threshold:",
    scan_median_quantile_threshold,
)

print(
    "Column-q90 quantile threshold:",
    scan_q90_quantile_threshold,
)

In [ ]:
intensity_masks = {
    "median_background_iqr": (
        column_median_smoothed
        > clean_background_median_threshold
    ),

    "median_scan_q75": (
        column_median_smoothed
        > scan_median_quantile_threshold
    ),

    "median_and_q90": (
        (
            column_median_smoothed
            > scan_median_quantile_threshold
        )
        & (
            column_q90_smoothed
            > scan_q90_quantile_threshold
        )
    ),

    "median_q90_reliable": (
        (
            column_median_smoothed
            > scan_median_quantile_threshold
        )
        & (
            column_q90_smoothed
            > scan_q90_quantile_threshold
        )
        & reliable_columns
    ),
}

for mask in intensity_masks.values():
    mask[
        :edge_margin
    ] = False

    mask[
        -edge_margin:
    ] = False

In [ ]:
def find_boolean_runs(
    mask: np.ndarray,
    target_value: bool,
) -> list[tuple[int, int]]:
    """
    Return inclusive runs containing the requested Boolean value.
    """
    mask = np.asarray(
        mask,
        dtype=bool,
    )

    runs = []
    start = None

    for index, value in enumerate(mask):
        if bool(value) == target_value:
            if start is None:
                start = index

        elif start is not None:
            runs.append(
                (
                    start,
                    index - 1,
                )
            )

            start = None

    if start is not None:
        runs.append(
            (
                start,
                mask.size - 1,
            )
        )

    return runs


def clean_column_mask(
    mask: np.ndarray,
    *,
    minimum_positive_run: int,
    maximum_negative_gap: int,
) -> np.ndarray:
    """
    Fill short internal gaps and remove short positive runs.
    """
    cleaned = np.asarray(
        mask,
        dtype=bool,
    ).copy()

    for start, end in find_boolean_runs(
        cleaned,
        False,
    ):
        touches_edge = (
            start == 0
            or end == cleaned.size - 1
        )

        gap_length = (
            end - start + 1
        )

        if (
            not touches_edge
            and gap_length
            <= maximum_negative_gap
        ):
            cleaned[
                start:
                end + 1
            ] = True

    for start, end in find_boolean_runs(
        cleaned,
        True,
    ):
        run_length = (
            end - start + 1
        )

        if (
            run_length
            < minimum_positive_run
        ):
            cleaned[
                start:
                end + 1
            ] = False

    return cleaned

In [ ]:
cleaned_intensity_masks = {
    name: clean_column_mask(
        mask,
        minimum_positive_run=(
            INTENSITY_SEGMENTATION_CONFIG[
                "minimum_positive_run"
            ]
        ),
        maximum_negative_gap=(
            INTENSITY_SEGMENTATION_CONFIG[
                "maximum_negative_gap"
            ]
        ),
    )
    for name, mask in intensity_masks.items()
}

In [ ]:
horizontal_positions = np.arange(
    image_width
)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(14, 10),
    sharex=True,
)

axes[0].plot(
    horizontal_positions,
    column_mean_smoothed,
    linewidth=1.2,
)

axes[0].set_title(
    "Smoothed column mean"
)

axes[0].set_ylabel(
    "Mean"
)

axes[1].plot(
    horizontal_positions,
    column_median_smoothed,
    linewidth=1.2,
)

axes[1].axhline(
    clean_background_median_threshold,
    linestyle="--",
    label="Clean median + IQR",
)

axes[1].axhline(
    scan_median_quantile_threshold,
    linestyle=":",
    label="Column median Q75",
)

axes[1].set_title(
    "Smoothed column median"
)

axes[1].set_ylabel(
    "Median"
)

axes[1].legend()

axes[2].plot(
    horizontal_positions,
    column_q90_smoothed,
    linewidth=1.2,
)

axes[2].axhline(
    scan_q90_quantile_threshold,
    linestyle="--",
    label="Column q90 Q75",
)

axes[2].set_title(
    "Smoothed column 90th percentile"
)

axes[2].set_xlabel(
    "Horizontal position"
)

axes[2].set_ylabel(
    "90th percentile"
)

axes[2].legend()

for axis in axes:
    axis.grid(
        alpha=0.25
    )

    axis.set_xlim(
        0,
        image_width - 1,
    )

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    len(
        cleaned_intensity_masks
    ) + 1,
    1,
    figsize=(14, 13),
    sharex=True,
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
)

axes[0].set_title(
    "Denoised sub-BM scan"
)

axes[0].set_ylabel(
    "Depth below BM"
)

for axis, (
    mask_name,
    mask,
) in zip(
    axes[1:],
    cleaned_intensity_masks.items(),
):
    axis.imshow(
        processed.denoised_scan,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            image_width - 0.5,
            processed.denoised_scan.shape[0]
            - 0.5,
            -0.5,
        ),
    )

    for start, end in find_boolean_runs(
        mask,
        True,
    ):
        axis.axvspan(
            start,
            end,
            color="tab:orange",
            alpha=0.30,
            linewidth=0,
        )

    axis.set_title(
        mask_name.replace(
            "_",
            " ",
        ).title()
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[-1].set_xlabel(
    "Horizontal position"
)

plt.tight_layout()
plt.show()

## Final Intensity-Only Hypertransmission Candidates

The initial intensity-mask comparison showed that column median intensity
provides the clearest broad localization of hypertransmission, while the
90th-percentile signal provides additional evidence that the upper-intensity
portion of the column is also elevated.

Thresholds are now defined consistently from the distributions of the
column-level signals:

$$
\tau_m
=
\operatorname{median}\{m(x)\}
+
k_m\operatorname{IQR}\{m(x)\},
$$

$$
\tau_{90}
=
\operatorname{median}\{q_{90}(x)\}
+
k_{90}\operatorname{IQR}\{q_{90}(x)\},
$$

where:

- $m(x)$ is the smoothed column median;
- $q_{90}(x)$ is the smoothed column 90th percentile;
- $\operatorname{IQR}$ is the interquartile range;
- $k_m$ and $k_{90}$ control threshold strictness.

Two candidate masks are retained:

1. **Sensitive hypertransmission mask** — column median exceeds its robust
   threshold.
2. **Conservative hypertransmission mask** — both column median and column
   90th percentile exceed their robust thresholds.

The retained-pixel fraction is recorded as a reliability diagnostic but is not
used as a hard exclusion criterion, because strongly structured columns may
naturally retain fewer pixels after structural masking.

In [ ]:
reference_column_medians = (
    column_median_smoothed[
        threshold_reference_mask
    ]
)

reference_column_q90 = (
    column_q90_smoothed[
        threshold_reference_mask
    ]
)

column_median_reference_median = float(
    np.median(
        reference_column_medians
    )
)

column_median_reference_iqr = float(
    np.quantile(
        reference_column_medians,
        0.75,
    )
    - np.quantile(
        reference_column_medians,
        0.25,
    )
)

column_q90_reference_median = float(
    np.median(
        reference_column_q90
    )
)

column_q90_reference_iqr = float(
    np.quantile(
        reference_column_q90,
        0.75,
    )
    - np.quantile(
        reference_column_q90,
        0.25,
    )
)

MEDIAN_IQR_MULTIPLIER = 1.0
Q90_IQR_MULTIPLIER = 0.5

median_robust_threshold = (
    column_median_reference_median
    + MEDIAN_IQR_MULTIPLIER
    * column_median_reference_iqr
)

q90_robust_threshold = (
    column_q90_reference_median
    + Q90_IQR_MULTIPLIER
    * column_q90_reference_iqr
)

print(
    "Column median reference median:",
    column_median_reference_median,
)

print(
    "Column median reference IQR:",
    column_median_reference_iqr,
)

print(
    "Column median robust threshold:",
    median_robust_threshold,
)

print(
    "Column q90 reference median:",
    column_q90_reference_median,
)

print(
    "Column q90 reference IQR:",
    column_q90_reference_iqr,
)

print(
    "Column q90 robust threshold:",
    q90_robust_threshold,
)

In [ ]:
candidate_hypertd_sensitive = (
    column_median_smoothed
    > median_robust_threshold
)

candidate_hypertd_conservative = (
    (
        column_median_smoothed
        > median_robust_threshold
    )
    & (
        column_q90_smoothed
        > q90_robust_threshold
    )
)

for mask in (
    candidate_hypertd_sensitive,
    candidate_hypertd_conservative,
):
    mask[
        :edge_margin
    ] = False

    mask[
        -edge_margin:
    ] = False

candidate_hypertd_sensitive = (
    clean_column_mask(
        candidate_hypertd_sensitive,
        minimum_positive_run=(
            INTENSITY_SEGMENTATION_CONFIG[
                "minimum_positive_run"
            ]
        ),
        maximum_negative_gap=(
            INTENSITY_SEGMENTATION_CONFIG[
                "maximum_negative_gap"
            ]
        ),
    )
)

candidate_hypertd_conservative = (
    clean_column_mask(
        candidate_hypertd_conservative,
        minimum_positive_run=(
            INTENSITY_SEGMENTATION_CONFIG[
                "minimum_positive_run"
            ]
        ),
        maximum_negative_gap=(
            INTENSITY_SEGMENTATION_CONFIG[
                "maximum_negative_gap"
            ]
        ),
    )
)

low_reliability_columns = (
    retained_fraction
    < INTENSITY_SEGMENTATION_CONFIG[
        "minimum_retained_fraction"
    ]
)

print(
    "Sensitive positive columns:",
    int(
        candidate_hypertd_sensitive.sum()
    ),
)

print(
    "Conservative positive columns:",
    int(
        candidate_hypertd_conservative.sum()
    ),
)

print(
    "Low-reliability columns:",
    int(
        low_reliability_columns.sum()
    ),
)

In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(14, 8),
    sharex=True,
)

axes[0].plot(
    horizontal_positions,
    column_median_smoothed,
    linewidth=1.2,
    label="Smoothed column median",
)

axes[0].axhline(
    median_robust_threshold,
    linestyle="--",
    linewidth=1.2,
    label=(
        "Median robust threshold"
    ),
)

axes[0].set_title(
    "Column median threshold"
)

axes[0].set_ylabel(
    "Median intensity"
)

axes[0].legend()
axes[0].grid(
    alpha=0.25
)

axes[1].plot(
    horizontal_positions,
    column_q90_smoothed,
    linewidth=1.2,
    label="Smoothed column 90th percentile",
)

axes[1].axhline(
    q90_robust_threshold,
    linestyle="--",
    linewidth=1.2,
    label=(
        "Q90 robust threshold"
    ),
)

axes[1].set_title(
    "Column 90th-percentile threshold"
)

axes[1].set_xlabel(
    "Horizontal position"
)

axes[1].set_ylabel(
    "90th-percentile intensity"
)

axes[1].legend()
axes[1].grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    3,
    1,
    figsize=(14, 10),
    sharex=True,
)

image_height, image_width = (
    processed.denoised_scan.shape
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
    extent=(
        -0.5,
        image_width - 0.5,
        image_height - 0.5,
        -0.5,
    ),
)

axes[0].set_title(
    "Denoised sub-BM scan"
)

axes[0].set_ylabel(
    "Depth below BM"
)

mask_panels = (
    (
        "Sensitive intensity-only hypertransmission mask",
        candidate_hypertd_sensitive,
        "tab:orange",
    ),
    (
        "Conservative intensity-only hypertransmission mask",
        candidate_hypertd_conservative,
        "tab:red",
    ),
)

for axis, (
    title,
    mask,
    color,
) in zip(
    axes[1:],
    mask_panels,
):
    axis.imshow(
        processed.denoised_scan,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            image_width - 0.5,
            image_height - 0.5,
            -0.5,
        ),
    )

    for start, end in find_boolean_runs(
        mask,
        True,
    ):
        axis.axvspan(
            start,
            end,
            color=color,
            alpha=0.30,
            linewidth=0,
        )

    axis.set_title(
        title
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[-1].set_xlabel(
    "Horizontal position"
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(
    figsize=(14, 4)
)

plt.plot(
    horizontal_positions,
    retained_fraction,
    linewidth=1.1,
    label="Retained pixel fraction",
)

plt.axhline(
    INTENSITY_SEGMENTATION_CONFIG[
        "minimum_retained_fraction"
    ],
    linestyle="--",
    label="Reliability reference",
)

plt.fill_between(
    horizontal_positions,
    0,
    1,
    where=low_reliability_columns,
    color="tab:gray",
    alpha=0.20,
    label="Low-reliability columns",
)

plt.title(
    "Structural-exclusion reliability diagnostic"
)

plt.xlabel(
    "Horizontal position"
)

plt.ylabel(
    "Retained pixel fraction"
)

plt.ylim(
    0,
    1.05,
)

plt.legend()
plt.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()
# use candidate hypter td conservative as the mask for now

## Depth Continuity Refinement

Intensity-only segmentation identifies columns with elevated brightness after
structural-pixel exclusion. Hypertransmission should also exhibit coherent
signal through depth rather than isolated bright tissue near the top of the
ROI.

Depth continuity is therefore calculated by comparing the horizontal intensity
pattern at one depth row with the pattern at a deeper row.

For a local horizontal neighborhood centered at $x$,

$$
C_z(x)
=
\operatorname{corr}
\left[
I(z, W_x),
I(z+\ell, W_x)
\right],
$$

where:

- $W_x$ is a local horizontal window;
- $z$ is depth below BM;
- $\ell$ is a selected depth lag;
- larger correlation indicates that the same horizontal pattern remains
  aligned through depth.

The column-level continuity signal is the median valid correlation across depth
pairs. It is used to refine candidate hypertransmission regions, not to define
brightness by itself.

In [ ]:
def compute_local_depth_continuity(
    image: np.ndarray,
    *,
    window_width: int = 15,
    depth_lag: int = 4,
    minimum_row_standard_deviation: float = 1e-6,
) -> np.ndarray:
    """
    Compute one depth-continuity value per horizontal image column.
    """
    image = np.asarray(
        image,
        dtype=np.float32,
    )

    if image.ndim != 2:
        raise ValueError(
            "image must be two-dimensional."
        )

    if window_width <= 1 or window_width % 2 == 0:
        raise ValueError(
            "window_width must be an odd integer greater than one."
        )

    if depth_lag <= 0 or depth_lag >= image.shape[0]:
        raise ValueError(
            "depth_lag must lie between 1 and image depth - 1."
        )

    radius = window_width // 2

    padded = np.pad(
        image,
        (
            (0, 0),
            (radius, radius),
        ),
        mode="reflect",
    )

    continuity = np.full(
        image.shape[1],
        np.nan,
        dtype=np.float32,
    )

    for x in range(
        image.shape[1]
    ):
        window = padded[
            :,
            x:
            x + window_width,
        ]

        correlations = []

        for depth in range(
            image.shape[0]
            - depth_lag
        ):
            first_row = window[
                depth
            ]

            second_row = window[
                depth + depth_lag
            ]

            if (
                np.std(first_row)
                < minimum_row_standard_deviation
                or np.std(second_row)
                < minimum_row_standard_deviation
            ):
                continue

            correlation = np.corrcoef(
                first_row,
                second_row,
            )[0, 1]

            if np.isfinite(correlation):
                correlations.append(
                    correlation
                )

        if correlations:
            continuity[x] = float(
                np.median(
                    correlations
                )
            )

    return continuity

In [ ]:
CONTINUITY_CONFIG = {
    "window_width": 15,
    "depth_lag": 4,
    "minimum_row_standard_deviation": 1e-6,
}

continuity_signal = (
    compute_local_depth_continuity(
        processed.denoised_scan,
        **CONTINUITY_CONFIG,
    )
)

continuity_smoothed = smooth_finite_signal(
    continuity_signal,
    sigma=2.0,
)

In [ ]:
fig, axes = plt.subplots(
    4,
    1,
    figsize=(14, 12),
    sharex=True,
)

axes[0].plot(
    horizontal_positions,
    column_median_smoothed,
)

axes[0].axhline(
    median_robust_threshold,
    linestyle="--",
)

axes[0].set_title(
    "Smoothed column median"
)

axes[1].plot(
    horizontal_positions,
    column_q90_smoothed,
)

axes[1].axhline(
    q90_robust_threshold,
    linestyle="--",
)

axes[1].set_title(
    "Smoothed column 90th percentile"
)

axes[2].plot(
    horizontal_positions,
    continuity_smoothed,
)

axes[2].set_title(
    "Local depth continuity"
)

axes[3].plot(
    horizontal_positions,
    retained_fraction,
)

axes[3].axhline(
    INTENSITY_SEGMENTATION_CONFIG[
        "minimum_retained_fraction"
    ],
    linestyle="--",
)

axes[3].set_title(
    "Retained structural-exclusion fraction"
)

axes[3].set_xlabel(
    "Horizontal position"
)

for axis in axes:
    axis.grid(
        alpha=0.25
    )

plt.tight_layout()
plt.show()

In [ ]:
continuity_reference = (
    continuity_smoothed[
        threshold_reference_mask
    ]
)

CONTINUITY_QUANTILES = {
    "q50": 0.50,
    "q60": 0.60,
    "q70": 0.70,
    "q80": 0.80,
}

continuity_thresholds = {
    label: float(
        np.quantile(
            continuity_reference,
            quantile,
        )
    )
    for label, quantile in (
        CONTINUITY_QUANTILES.items()
    )
}

for label, value in (
    continuity_thresholds.items()
):
    print(
        f"{label}: {value:.4f}"
    )

In [ ]:
continuity_refined_masks = {}

for label, threshold_value in (
    continuity_thresholds.items()
):
    continuity_mask = (
        continuity_smoothed
        > threshold_value
    )

    refined_mask = (
        candidate_hypertd_conservative
        & continuity_mask
    )

    refined_mask = clean_column_mask(
        refined_mask,
        minimum_positive_run=5,
        maximum_negative_gap=2,
    )

    continuity_refined_masks[
        label
    ] = refined_mask

The sensitive mask detects 83 columns, while the conservative median-plus-$q_{90}$ mask detects 75. Their similarity means the column median is doing most of the localization, while the $q_{90}$ requirement removes only a few weaker positions.

The small detection around $x\approx380$ appears in both masks. It may be a legitimate isolated bright feature, a vessel-related structure, or a false positive. Depth continuity should help characterize it.

The low-reliability columns cluster around strong structural boundaries. This confirms that retained fraction should remain a diagnostic rather than a hard negative gate.

### Continuity-Refined Hypertransmission Masks

The conservative intensity-only mask is refined using several depth-continuity
thresholds.

For a continuity threshold $\tau_C$, the refined mask is

$$
M_{\mathrm{refined}}(x)
=
M_{\mathrm{intensity}}(x)
\land
\mathbf{1}
\left[
C(x)>\tau_C
\right],
$$

where:

- $M_{\mathrm{intensity}}(x)$ is the conservative median-plus-$q_{90}$
  hypertransmission mask;
- $C(x)$ is the smoothed local depth-continuity signal;
- $\tau_C$ is selected from the scan-level continuity distribution.

Multiple thresholds are compared to determine whether continuity removes
isolated candidates without fragmenting the main hypertransmission region.

In [ ]:
continuity_refined_masks = {}

for label, threshold_value in (
    continuity_thresholds.items()
):
    continuity_positive = (
        continuity_smoothed
        > threshold_value
    )

    refined_mask = (
        candidate_hypertd_conservative
        & continuity_positive
    )

    refined_mask[:edge_margin] = False
    refined_mask[-edge_margin:] = False

    refined_mask = clean_column_mask(
        refined_mask,
        minimum_positive_run=(
            INTENSITY_SEGMENTATION_CONFIG[
                "minimum_positive_run"
            ]
        ),
        maximum_negative_gap=(
            INTENSITY_SEGMENTATION_CONFIG[
                "maximum_negative_gap"
            ]
        ),
    )

    continuity_refined_masks[
        label
    ] = refined_mask

    print(
        f"{label}: "
        f"{int(refined_mask.sum())} positive columns"
    )

In [ ]:
fig, axis = plt.subplots(
    figsize=(14, 5)
)

axis.plot(
    horizontal_positions,
    continuity_smoothed,
    linewidth=1.3,
    label="Depth continuity",
)

for label, threshold_value in (
    continuity_thresholds.items()
):
    axis.axhline(
        threshold_value,
        linestyle="--",
        linewidth=1.0,
        label=(
            f"{label.upper()} = "
            f"{threshold_value:.3f}"
        ),
    )

for start, end in find_boolean_runs(
    candidate_hypertd_conservative,
    True,
):
    axis.axvspan(
        start,
        end,
        color="tab:red",
        alpha=0.12,
    )

axis.set_title(
    "Depth continuity within conservative intensity candidates"
)

axis.set_xlabel(
    "Horizontal position"
)

axis.set_ylabel(
    "Continuity"
)

axis.set_xlim(
    0,
    image_width - 1,
)

axis.grid(
    alpha=0.25
)

axis.legend(
    ncols=2
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    len(continuity_refined_masks) + 2,
    1,
    figsize=(14, 16),
    sharex=True,
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
    extent=(
        -0.5,
        image_width - 0.5,
        image_height - 0.5,
        -0.5,
    ),
)

axes[0].set_title(
    "Denoised sub-BM scan"
)

axes[0].set_ylabel(
    "Depth below BM"
)

axes[1].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
    extent=(
        -0.5,
        image_width - 0.5,
        image_height - 0.5,
        -0.5,
    ),
)

for start, end in find_boolean_runs(
    candidate_hypertd_conservative,
    True,
):
    axes[1].axvspan(
        start,
        end,
        color="tab:red",
        alpha=0.30,
        linewidth=0,
    )

axes[1].set_title(
    "Conservative intensity-only candidate"
)

axes[1].set_ylabel(
    "Depth below BM"
)

for axis, (
    label,
    mask,
) in zip(
    axes[2:],
    continuity_refined_masks.items(),
):
    axis.imshow(
        processed.denoised_scan,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            image_width - 0.5,
            image_height - 0.5,
            -0.5,
        ),
    )

    for start, end in find_boolean_runs(
        mask,
        True,
    ):
        axis.axvspan(
            start,
            end,
            color="tab:blue",
            alpha=0.30,
            linewidth=0,
        )

    axis.set_title(
        f"Continuity refinement: "
        f"{label.upper()} "
        f"(threshold="
        f"{continuity_thresholds[label]:.3f})"
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[-1].set_xlabel(
    "Horizontal position"
)

plt.tight_layout()
plt.show()

In [ ]:
continuity_comparison_masks = {}

SELECTED_CONTINUITY_LEVEL = "q60"

selected_continuity_threshold = (
    continuity_thresholds[
        SELECTED_CONTINUITY_LEVEL
    ]
)

continuity_positive = (
    continuity_smoothed
    > selected_continuity_threshold
)

for name, intensity_mask in {
    "sensitive": (
        candidate_hypertd_sensitive
    ),
    "conservative": (
        candidate_hypertd_conservative
    ),
}.items():
    refined = (
        intensity_mask
        & continuity_positive
    )

    refined = clean_column_mask(
        refined,
        minimum_positive_run=5,
        maximum_negative_gap=2,
    )

    continuity_comparison_masks[
        name
    ] = refined

In [ ]:
fig, axes = plt.subplots(
    4,
    1,
    figsize=(14, 12),
    sharex=True,
)

comparison_panels = (
    (
        "Sensitive intensity only",
        candidate_hypertd_sensitive,
        "tab:orange",
    ),
    (
        "Sensitive + continuity",
        continuity_comparison_masks[
            "sensitive"
        ],
        "tab:blue",
    ),
    (
        "Conservative intensity only",
        candidate_hypertd_conservative,
        "tab:red",
    ),
    (
        "Conservative + continuity",
        continuity_comparison_masks[
            "conservative"
        ],
        "tab:purple",
    ),
)

for axis, (
    title,
    mask,
    color,
) in zip(
    axes,
    comparison_panels,
):
    axis.imshow(
        processed.denoised_scan,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            image_width - 0.5,
            image_height - 0.5,
            -0.5,
        ),
    )

    for start, end in find_boolean_runs(
        mask,
        True,
    ):
        axis.axvspan(
            start,
            end,
            color=color,
            alpha=0.30,
            linewidth=0,
        )

    axis.set_title(
        title
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[-1].set_xlabel(
    "Horizontal position"
)

plt.tight_layout()
plt.show()

In [ ]:
candidate_hypertd_region = (
    candidate_hypertd_conservative
)

candidate_hypertd_continuous = (
    continuity_refined_masks["q60"]
)

In [ ]:
SELECTED_CONTINUITY_LEVEL = "q60"

selected_continuity_threshold = (
    continuity_thresholds[
        SELECTED_CONTINUITY_LEVEL
    ]
)

candidate_hypertd_region = (
    candidate_hypertd_conservative.copy()
)

candidate_hypertd_continuous = (
    continuity_refined_masks[
        SELECTED_CONTINUITY_LEVEL
    ].copy()
)

print(
    "Selected continuity level:",
    SELECTED_CONTINUITY_LEVEL,
)

print(
    "Selected continuity threshold:",
    selected_continuity_threshold,
)

print(
    "Intensity-only candidate columns:",
    int(
        candidate_hypertd_region.sum()
    ),
)

print(
    "Continuity-refined columns:",
    int(
        candidate_hypertd_continuous.sum()
    ),
)

## Column-Level Vertical Organization

The structural mask was previously used only to exclude strongly oriented
pixels when estimating the choroidal intensity distribution.

Vertical organization is now reintroduced as a positive feature. For each
horizontal column, the following summaries are calculated across depth:

$$
\overline V(x)
=
\operatorname{mean}_z V(z,x),
$$

$$
V_{90}(x)
=
Q_{0.90,z}\{V(z,x)\},
$$

and

$$
F_V(x)
=
\frac{1}{D}
\sum_z
\mathbf{1}
\left[
V(z,x)\geq\tau_V
\land
G(z,x)\geq\tau_G
\right].
$$

The final quantity is the fraction of the column occupied by pixels that are
both strongly vertical and sufficiently high-gradient.

These signals are evaluated only after intensity-based hypertransmission and
depth continuity have been established. They therefore refine persistent
hypertransmission into candidate barcoding rather than identifying bright or
vertical structures independently.

In [ ]:
column_verticality_mean = np.mean(
    simple_verticality_map,
    axis=0,
).astype(np.float32)

column_verticality_q90 = np.quantile(
    simple_verticality_map,
    0.90,
    axis=0,
).astype(np.float32)

column_strong_vertical_fraction = np.mean(
    structural_mask,
    axis=0,
).astype(np.float32)

column_verticality_mean_smoothed = (
    smooth_finite_signal(
        column_verticality_mean,
        sigma=2.0,
    )
)

column_verticality_q90_smoothed = (
    smooth_finite_signal(
        column_verticality_q90,
        sigma=2.0,
    )
)

column_strong_vertical_fraction_smoothed = (
    smooth_finite_signal(
        column_strong_vertical_fraction,
        sigma=2.0,
    )
)

In [ ]:
fig, axes = plt.subplots(
    4,
    1,
    figsize=(14, 12),
    sharex=True,
)

axes[0].plot(
    horizontal_positions,
    column_verticality_mean_smoothed,
)

axes[0].set_title(
    "Mean column verticality"
)

axes[0].set_ylabel(
    "Mean verticality"
)

axes[1].plot(
    horizontal_positions,
    column_verticality_q90_smoothed,
)

axes[1].set_title(
    "Column 90th-percentile verticality"
)

axes[1].set_ylabel(
    "Verticality Q90"
)

axes[2].plot(
    horizontal_positions,
    column_strong_vertical_fraction_smoothed,
)

axes[2].set_title(
    "Strong vertical-pixel fraction"
)

axes[2].set_ylabel(
    "Depth fraction"
)

axes[3].plot(
    horizontal_positions,
    continuity_smoothed,
    label="Depth continuity",
)

for start, end in find_boolean_runs(
    candidate_hypertd_region,
    True,
):
    axes[3].axvspan(
        start,
        end,
        color="tab:red",
        alpha=0.12,
    )

for start, end in find_boolean_runs(
    candidate_hypertd_continuous,
    True,
):
    axes[3].axvspan(
        start,
        end,
        color="tab:blue",
        alpha=0.18,
    )

axes[3].set_title(
    "Continuity with intensity and continuity candidates"
)

axes[3].set_xlabel(
    "Horizontal position"
)

axes[3].set_ylabel(
    "Continuity"
)

for axis in axes:
    axis.grid(
        alpha=0.25
    )

plt.tight_layout()
plt.show()

In [ ]:
vertical_fraction_reference = (
    column_strong_vertical_fraction_smoothed[
        threshold_reference_mask
    ]
)

VERTICAL_FRACTION_QUANTILES = {
    "q50": 0.50,
    "q60": 0.60,
    "q70": 0.70,
    "q80": 0.80,
}

vertical_fraction_thresholds = {
    label: float(
        np.quantile(
            vertical_fraction_reference,
            quantile,
        )
    )
    for label, quantile in (
        VERTICAL_FRACTION_QUANTILES.items()
    )
}

for label, threshold_value in (
    vertical_fraction_thresholds.items()
):
    print(
        f"{label}: "
        f"{threshold_value:.4f}"
    )

In [ ]:
candidate_barcode_masks = {}

for label, threshold_value in (
    vertical_fraction_thresholds.items()
):
    vertical_positive = (
        column_strong_vertical_fraction_smoothed
        > threshold_value
    )

    barcode_mask = (
        candidate_hypertd_continuous
        & vertical_positive
    )

    barcode_mask[:edge_margin] = False
    barcode_mask[-edge_margin:] = False

    barcode_mask = clean_column_mask(
        barcode_mask,
        minimum_positive_run=5,
        maximum_negative_gap=2,
    )

    candidate_barcode_masks[
        label
    ] = barcode_mask

    print(
        f"{label}: "
        f"{int(barcode_mask.sum())} "
        "candidate barcode columns"
    )

In [ ]:
fig, axes = plt.subplots(
    len(candidate_barcode_masks) + 2,
    1,
    figsize=(14, 16),
    sharex=True,
)

axes[0].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
    extent=(
        -0.5,
        image_width - 0.5,
        image_height - 0.5,
        -0.5,
    ),
)

axes[0].set_title(
    "Denoised sub-BM scan"
)

axes[1].imshow(
    processed.denoised_scan,
    cmap="gray",
    aspect="auto",
    extent=(
        -0.5,
        image_width - 0.5,
        image_height - 0.5,
        -0.5,
    ),
)

for start, end in find_boolean_runs(
    candidate_hypertd_continuous,
    True,
):
    axes[1].axvspan(
        start,
        end,
        color="tab:blue",
        alpha=0.30,
        linewidth=0,
    )

axes[1].set_title(
    "Continuity-refined hypertransmission"
)

for axis, (
    label,
    mask,
) in zip(
    axes[2:],
    candidate_barcode_masks.items(),
):
    axis.imshow(
        processed.denoised_scan,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            image_width - 0.5,
            image_height - 0.5,
            -0.5,
        ),
    )

    for start, end in find_boolean_runs(
        mask,
        True,
    ):
        axis.axvspan(
            start,
            end,
            color="tab:purple",
            alpha=0.30,
            linewidth=0,
        )

    axis.set_title(
        f"Vertical-fraction refinement: "
        f"{label.upper()} "
        f"(threshold="
        f"{vertical_fraction_thresholds[label]:.3f})"
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[-1].set_xlabel(
    "Horizontal position"
)

plt.tight_layout()
plt.show()

## Provisional Candidate Barcoding Mask

The provisional configuration selected from the current scan is:

- structural exclusion using verticality with a gradient-magnitude threshold
  at the 80th percentile;
- robust column-median and column-90th-percentile intensity thresholds;
- depth-continuity refinement at the 60th percentile;
- strong vertical-pixel fraction refinement at the 70th percentile.

The resulting candidate mask requires three forms of evidence:

$$
M_{\mathrm{barcode}}(x)
=
M_{\mathrm{intensity}}(x)
\land
M_{\mathrm{continuity}}(x)
\land
M_{\mathrm{vertical}}(x).
$$

This remains an exploratory configuration. Its reliability must be evaluated
across multiple subjects and B-scans before the thresholds are fixed.

In [ ]:
SELECTED_CONTINUITY_LEVEL = "q60"
SELECTED_VERTICAL_FRACTION_LEVEL = "q70"

selected_continuity_threshold = (
    continuity_thresholds[
        SELECTED_CONTINUITY_LEVEL
    ]
)

selected_vertical_fraction_threshold = (
    vertical_fraction_thresholds[
        SELECTED_VERTICAL_FRACTION_LEVEL
    ]
)

provisional_hypertd_mask = (
    continuity_refined_masks[
        SELECTED_CONTINUITY_LEVEL
    ].copy()
)

provisional_barcode_mask = (
    candidate_barcode_masks[
        SELECTED_VERTICAL_FRACTION_LEVEL
    ].copy()
)

print(
    "Continuity level:",
    SELECTED_CONTINUITY_LEVEL,
)

print(
    "Continuity threshold:",
    selected_continuity_threshold,
)

print(
    "Vertical-fraction level:",
    SELECTED_VERTICAL_FRACTION_LEVEL,
)

print(
    "Vertical-fraction threshold:",
    selected_vertical_fraction_threshold,
)

print(
    "Continuity-refined hypertransmission columns:",
    int(
        provisional_hypertd_mask.sum()
    ),
)

print(
    "Provisional candidate barcode columns:",
    int(
        provisional_barcode_mask.sum()
    ),
)

In [ ]:
fig, axes = plt.subplots(
    5,
    1,
    figsize=(14, 15),
    sharex=True,
)

stage_panels = (
    (
        "Denoised sub-BM scan",
        None,
        None,
    ),
    (
        "Conservative intensity candidate",
        candidate_hypertd_conservative,
        "tab:red",
    ),
    (
        "Q60 continuity-refined hypertransmission",
        provisional_hypertd_mask,
        "tab:blue",
    ),
    (
        "Q70 vertical-fraction refinement",
        provisional_barcode_mask,
        "tab:purple",
    ),
)

for axis, (
    title,
    mask,
    color,
) in zip(
    axes[:4],
    stage_panels,
):
    axis.imshow(
        processed.denoised_scan,
        cmap="gray",
        aspect="auto",
        extent=(
            -0.5,
            image_width - 0.5,
            image_height - 0.5,
            -0.5,
        ),
    )

    if mask is not None:
        for start, end in find_boolean_runs(
            mask,
            True,
        ):
            axis.axvspan(
                start,
                end,
                color=color,
                alpha=0.30,
                linewidth=0,
            )

    axis.set_title(
        title
    )

    axis.set_ylabel(
        "Depth below BM"
    )

axes[4].plot(
    horizontal_positions,
    continuity_smoothed,
    label="Depth continuity",
)

axes[4].axhline(
    selected_continuity_threshold,
    linestyle="--",
    label=(
        f"Q60 continuity = "
        f"{selected_continuity_threshold:.3f}"
    ),
)

axes[4].plot(
    horizontal_positions,
    column_strong_vertical_fraction_smoothed,
    label="Strong vertical fraction",
)

axes[4].axhline(
    selected_vertical_fraction_threshold,
    linestyle=":",
    label=(
        f"Q70 vertical fraction = "
        f"{selected_vertical_fraction_threshold:.3f}"
    ),
)

for start, end in find_boolean_runs(
    provisional_barcode_mask,
    True,
):
    axes[4].axvspan(
        start,
        end,
        color="tab:purple",
        alpha=0.15,
    )

axes[4].set_title(
    "Final refinement signals"
)

axes[4].set_xlabel(
    "Horizontal position"
)

axes[4].set_ylabel(
    "Signal value"
)

axes[4].legend(
    ncols=2
)

axes[4].grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
provisional_barcode_intervals = (
    find_boolean_runs(
        provisional_barcode_mask,
        True,
    )
)

print(
    "Number of provisional candidate intervals:",
    len(
        provisional_barcode_intervals
    ),
)

for interval_number, (
    start,
    end,
) in enumerate(
    provisional_barcode_intervals,
    start=1,
):
    width = end - start + 1

    print(
        f"{interval_number:02d}. "
        f"x={start}–{end}, "
        f"width={width} px"
    )

 Now we hold the settings fix and test how well it can generalize to the all 10 randomly selected scans.

 Config:
 ```
SELECTED_STRUCTURAL_MASK = "gradient_q80"
SELECTED_CONTINUITY_LEVEL = "q60"
SELECTED_VERTICAL_FRACTION_LEVEL = "q70"
MEDIAN_IQR_MULTIPLIER = 1.0
Q90_IQR_MULTIPLIER = 0.5
 ```